# Dana's Bakery: deterministic order processing with LangGraph

This Google Colab notebook builds a small order-processing graph with explicit Python rules. It uses no LLM and requires no API keys. Run the cells from top to bottom. The graph tracks its working state internally and returns only the six fields in the output contract.


In [ ]:
%pip -q install langgraph


## Runtime assumptions

Use a Python 3 Google Colab runtime with internet access for the initial `langgraph` install and Mermaid PNG rendering. The visualization cell calls LangGraph's `draw_mermaid_png()` renderer.


In [ ]:
from typing import TypedDict
from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph

MENU = {
    'challah': {'price': 18, 'stock': 10},
    'sourdough': {'price': 25, 'stock': 4},
    'rugelach': {'price': 30, 'stock': 3},
    'babka': {'price': 45, 'stock': 0},
}
SUBSTITUTION_CHAIN = ['babka', 'rugelach', 'sourdough', 'challah']
OUTPUT_KEYS = ('status', 'item', 'delivery', 'total', 'message', 'steps')


## State schemas

`InputState` describes the caller's four fields. `OrderState` adds the internal fields needed to process the order. `OutputState` filters the compiled graph result so `invoke()` returns exactly six keys; quantity, day, city, and substitution cursor stay internal.


In [ ]:
class InputState(TypedDict):
    item: str
    quantity: int
    day: str
    city: str


class OrderState(TypedDict):
    item: str
    requested_item: str
    quantity: int
    day: str
    city: str
    status: str
    delivery: str
    total: float
    message: str
    steps: list[str]
    attempt_count: int
    candidate_index: int


class OutputState(TypedDict):
    status: str
    item: str
    delivery: str
    total: float
    message: str
    steps: list[str]


## Processing nodes

The substitution node advances through the chain from the requested product. It accepts the first candidate whose stock covers the entire order. `attempt_count` tracks substitute candidates checked, up to two; rejection occurs only when both allowed candidates are unavailable.


In [ ]:
def validate_order(state: OrderState) -> dict:
    requested_item = state['item'].strip().lower()
    quantity = state['quantity']
    steps = [f'Received {quantity} x {requested_item or "(blank item)"}.']

    if requested_item not in MENU:
        return {
            'item': requested_item, 'status': 'rejected', 'delivery': '', 'total': 0.0,
            'message': 'Sorry, that item is not on our menu. Please choose a listed bakery item.',
            'steps': steps + ['Rejected: item is not on the menu.'],
        }
    if quantity <= 0:
        return {
            'item': requested_item, 'status': 'rejected', 'delivery': '', 'total': 0.0,
            'message': 'Sorry, quantity must be at least 1. Please update your order.',
            'steps': steps + ['Rejected: quantity must be positive.'],
        }

    return {
        'item': requested_item, 'requested_item': requested_item, 'status': 'accepted', 'delivery': '', 'total': 0.0,
        'message': '', 'steps': steps + ['Validation passed.'],
        'attempt_count': 0, 'candidate_index': SUBSTITUTION_CHAIN.index(requested_item),
    }


def route_after_validation(state: OrderState) -> str:
    return 'check_stock' if state['status'] == 'accepted' else END


def check_stock(state: OrderState) -> dict:
    item = state['item']
    quantity = state['quantity']
    steps = list(state['steps'])

    if MENU[item]['stock'] >= quantity:
        if item != state['requested_item']:
            steps.append(f'Substituted with {item}.')
        else:
            steps.append(f'{item} stock covers the requested quantity.')
        return {'status': 'accepted', 'steps': steps}

    next_index = state['candidate_index'] + 1
    attempts = state['attempt_count']
    steps.append(f'{item} stock is insufficient.')

    if attempts >= 2:
        return {
            'status': 'rejected', 'delivery': '', 'total': 0.0,
            'message': 'We are sorry, but we could not find an available substitute after two attempts.',
            'steps': steps + ['Rejected after the second unavailable substitution candidate.'],
        }
    if next_index >= len(SUBSTITUTION_CHAIN):
        return {
            'status': 'rejected', 'delivery': '', 'total': 0.0,
            'message': 'We are sorry, but we could not fulfill this order or find an available substitute.',
            'steps': steps + ['Rejected: no later substitution candidate exists.'],
        }

    next_item = SUBSTITUTION_CHAIN[next_index]
    attempts += 1
    steps.append(f'Substitution attempt {attempts}: checking {next_item}.')
    return {
        'item': next_item, 'candidate_index': next_index,
        'attempt_count': attempts, 'steps': steps,
    }


def route_after_stock(state: OrderState) -> str:
    if state['status'] == 'rejected':
        return END
    if MENU[state['item']]['stock'] >= state['quantity']:
        return 'fulfill_order'
    return 'check_stock'


def fulfill_order(state: OrderState) -> dict:
    item = state['item']
    quantity = state['quantity']
    city = state['city'].strip()
    day = state['day'].strip().lower()
    steps = list(state['steps'])
    baked_goods_subtotal = float(MENU[item]['price'] * quantity)

    if city.lower() == 'jerusalem':
        delivery = 'delivery'
        delivery_fee = 20.0
        fulfillment_note = 'Your order will be delivered in Jerusalem for 20 NIS.'
        steps.append('Delivery selected: Jerusalem delivery fee is 20 NIS.')
    else:
        delivery = 'pickup'
        delivery_fee = 0.0
        location = city if city else 'no city provided'
        fulfillment_note = f'Delivery is unavailable for {location}; your order is set for pickup.'
        steps.append('Pickup selected: delivery is unavailable outside Jerusalem or no city was provided.')

    if baked_goods_subtotal > 150 and day != 'friday':
        discount = baked_goods_subtotal * 0.15
        discounted_subtotal = baked_goods_subtotal - discount
        steps.append(f'Avi discount applied: 15% off baked goods ({discount:.2f} NIS); delivery is not discounted.')
        discount_note = f' A 15% discount was applied to baked goods ({discount:.2f} NIS).'
    else:
        discounted_subtotal = baked_goods_subtotal
        if baked_goods_subtotal > 150 and day == 'friday':
            steps.append('No discount applied: Avi’s rule excludes Friday.')
            discount_note = ' No discount applies on Friday.'
        else:
            steps.append('No discount applied: baked-goods subtotal is not over 150 NIS.')
            discount_note = ''

    total = float(discounted_subtotal + delivery_fee)
    message = f'Order accepted: {quantity} x {item}. {fulfillment_note}{discount_note} Total: {total:.2f} NIS.'
    steps.append(f'Final total: {total:.2f} NIS.')
    return {'status': 'accepted', 'delivery': delivery, 'total': total, 'message': message, 'steps': steps}


## Build and visualize the graph

In [ ]:
builder = StateGraph(OrderState, input=InputState, output=OutputState)
builder.add_node('validate_order', validate_order)
builder.add_node('check_stock', check_stock)
builder.add_node('fulfill_order', fulfill_order)
builder.add_edge(START, 'validate_order')
builder.add_conditional_edges('validate_order', route_after_validation, {'check_stock': 'check_stock', END: END})
builder.add_conditional_edges('check_stock', route_after_stock, {'check_stock': 'check_stock', 'fulfill_order': 'fulfill_order', END: END})
builder.add_edge('fulfill_order', END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))


## Example invocation

In [ ]:
sample_order = {'item': 'challah', 'quantity': 2, 'day': 'Sunday', 'city': ''}
sample_result = graph.invoke(sample_order)
sample_result


## Deterministic path checks

The examples cover normal fulfillment, Jerusalem delivery, substitution, both validation failures, the two-attempt substitution limit, and the Friday discount exception. Assertions check the six-key output contract and important path-specific results.


In [ ]:
def run_case(name: str, order: dict, expected: dict) -> dict:
    result = graph.invoke(order)
    assert set(result) == set(OUTPUT_KEYS), f'{name}: unexpected output keys {set(result)}'
    for key, value in expected.items():
        assert result[key] == value, f'{name}: {key} expected {value!r}, got {result[key]!r}'
    print(f'PASS — {name}: {result["status"]}, {result["item"]}, {result["delivery"]}, {result["total"]:.2f} NIS')
    return result

cases = {}
cases['valid pickup'] = run_case(
    'valid pickup', {'item': 'challah', 'quantity': 2, 'day': 'Sunday', 'city': ''},
    {'status': 'accepted', 'item': 'challah', 'delivery': 'pickup', 'total': 36.0},
)
cases['Jerusalem delivery'] = run_case(
    'Jerusalem delivery', {'item': 'sourdough', 'quantity': 2, 'day': 'Monday', 'city': 'Jerusalem'},
    {'status': 'accepted', 'item': 'sourdough', 'delivery': 'delivery', 'total': 70.0},
)
cases['successful substitution'] = run_case(
    'successful substitution', {'item': 'babka', 'quantity': 4, 'day': 'Monday', 'city': 'Jerusalem'},
    {'status': 'accepted', 'item': 'sourdough', 'delivery': 'delivery', 'total': 120.0},
)
assert any('Substituted with sourdough' in step for step in cases['successful substitution']['steps'])
cases['invalid item'] = run_case(
    'invalid item', {'item': 'croissant', 'quantity': 1, 'day': 'Tuesday', 'city': 'Jerusalem'},
    {'status': 'rejected', 'item': 'croissant', 'delivery': '', 'total': 0.0},
)
cases['non-positive quantity'] = run_case(
    'non-positive quantity', {'item': 'challah', 'quantity': 0, 'day': 'Tuesday', 'city': ''},
    {'status': 'rejected', 'item': 'challah', 'delivery': '', 'total': 0.0},
)
cases['two substitution failures'] = run_case(
    'two substitution failures', {'item': 'babka', 'quantity': 5, 'day': 'Tuesday', 'city': 'Jerusalem'},
    {'status': 'rejected', 'item': 'sourdough', 'delivery': '', 'total': 0.0},
)
assert any('after two attempts' in cases['two substitution failures']['message'] for _ in [0])
cases['Friday discount exception'] = run_case(
    'Friday discount exception', {'item': 'challah', 'quantity': 9, 'day': 'Friday', 'city': 'Jerusalem'},
    {'status': 'accepted', 'item': 'challah', 'delivery': 'delivery', 'total': 182.0},
)
cases['non-Friday discount'] = run_case(
    'non-Friday discount', {'item': 'challah', 'quantity': 9, 'day': 'Thursday', 'city': 'Jerusalem'},
    {'status': 'accepted', 'item': 'challah', 'delivery': 'delivery', 'total': 157.7},
)
